## 09.1.1 Vectorisation

```
date: "Block 09"
author: "Daniel Lawson"
email: dan.lawson@bristol.ac.uk
output: html_document
version: 4.0.0
```

## Portfolio 09.1.1 - Vectorisation in pytorch

* Why has modern machine learning converged on tensor operations (matrix products, convolutions, element-wise functions)?
    * Link vectorisation to automatic differentiation, and how GPUs can compute this "quickly".
    
# 0. Introduction

Here we will put to practice the ideas described in the lecture, notably:

* Vectorisation,
* And its uses in pytorch for GPU calculation and **automatic differentiation**
* This is built upon the [kaggle autograd tutorial](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html)

## 0.1 Prerequisites:

In [1]:
## Install requirements, e.g.: 
# !pip install numpy
# !pip install pytorch

In [2]:
!pipreqsnb --savepath requirements-block09.txt block09.1.1-Vectorisation.ipynb

pipreqs  --savepath requirements-block09.txt /Users/madjl/teach/dst-private/Workshops/__temp_pipreqsnb_folder
INFO: Not scanning for jupyter notebooks.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in requirements-block09.txt


In [3]:
print("\nREQUIREMENTS:\n")
with open('requirements-block09.txt', 'r') as f:
    print(f.read())


REQUIREMENTS:

numpy==2.5.4
torch==2.9.0



## 1. Vectorisation

Vectorisation is straightforward in python (and R). You should always try to code with vectors/arrays, though For loops are sometimes necessary.

In python:

* For loops may not be intrinsically worse, but they encourage poor coding practice.

In R: 

* For loops are very inefficient. For efficiency you may have to go out of your way to vectorise.

In C/C++:

* OpenMP allows for loops to be parallelised without any additional effort - just remember to avoid using the results of previous loops.
* Modern updates (from C++-11) include many explicit vectorisations, allowing map/reduce vectorisations to be exploited directly.

### 1.1 Arrays in Python

First, an array representation reminder in python:

In [4]:
import numpy as np
import torch

In [5]:
a = np.array([[1,2],[3,4]])
b = np.array([[1,1],[1,1]])
print("a =")
print(a)
print("b =")
print(b)

a =
[[1 2]
 [3 4]]
b =
[[1 1]
 [1 1]]


In [6]:
# substraction and addition

print("a + b =")
print(a + b)
print("a - b =")
print(a - b)

a + b =
[[2 3]
 [4 5]]
a - b =
[[0 1]
 [2 3]]


In [7]:
# Element wise multiplication
print("a * b =")
print(a*b)

a * b =
[[1 2]
 [3 4]]


In [8]:
# Matrix multiplication
print("a@b =")
print(a@b)
print("np.dot(a,b) =")
print(np.dot(a,b))

a@b =
[[3 3]
 [7 7]]
np.dot(a,b) =
[[3 3]
 [7 7]]


In [9]:
# numpy array with one row
c =  np.array([1,2,3])
print("c.shape =")
print(c.shape)
# numpy array with three rows
d = np.array([[1],[2],[3]])
print("d.shape")
print(d.shape)

c.shape =
(3,)
d.shape
(3, 1)


In [10]:
# Define an 3x3 2d array
e = np.array([[1,2,3],[4,5,6],[7,8,9]])
print(e)
print("first element in the array, e[0,0] =")
print(e[0,0])
print("first row of the array. e[0,:] =")
print(e[0,:])
print("second column of the array. e[:,1] =")
print(e[:,1])

[[1 2 3]
 [4 5 6]
 [7 8 9]]
first element in the array, e[0,0] =
1
first row of the array. e[0,:] =
[1 2 3]
second column of the array. e[:,1] =
[2 5 8]


### 1.2 Comparing loops to vectorisations

Now we'll make a simulation to compare between vectorised and non-vectorised code. This is just a simple matrix times vector computation.


In [11]:
#create some test data and simulate results
N=100000 # Number of rows
K=200 # Number of columns
X = np.random.randn(N,K)
W = np.random.rand(K)

In [12]:
def matrix_times_vector_forloop(X,W):
    N,K=X.shape
    # Initialize theta
    forloop = []
    for i in range(N):
        hypo_i = 0
        for j in range(K):
            hypo_i += W[j]*X[i,j]
        forloop.append(hypo_i)
    return(forloop)

In [13]:
%%time
forloop=matrix_times_vector_forloop(X,W)

CPU times: user 3.47 s, sys: 138 ms, total: 3.6 s
Wall time: 4.9 s


In [14]:
%%time
# matrix format
vect = X@W

CPU times: user 7.53 ms, sys: 21.7 ms, total: 29.2 ms
Wall time: 33.7 ms


In [15]:
## Check the answers
print("for loop")
print(forloop[0:4])
print("vectorised")
print(vect[0:4])

for loop
[np.float64(-6.390934820289989), np.float64(2.596838083531284), np.float64(2.70073216700299), np.float64(-4.650438611166893)]
vectorised
[-6.39093482  2.59683808  2.70073217 -4.65043861]


CHALLENGE:  Make a plot of how the two approaches change in performance as a function of N (and/or K). What is the computational scaling?

See https://towardsdatascience.com/vectorization-implementation-in-machine-learning-ca652920c55d for an example that is much more extreme.

# 2. PyTorch

At its core, [PyTorch](https://pytorch.org/) provides two main features:

- An n-dimensional **tensor**, similar to a [NumPy](https://numpy.org/) array but able to run on GPUs.
- **Automatic differentiation**: the gradient of anything computed from tensor operations is worked out for you.

Neural networks are the famous use case, but nothing here is specific to them: any model whose loss can be written as tensor operations can be fitted the same way.

We will use a running example of fitting $y=\sin(x)$ with a third-order polynomial $y = a + bx + cx^2 + dx^3$, choosing the four parameters by gradient descent to minimise the squared error between prediction and truth.

## PyTorch vs Numpy

- Like numpy arrays, tensors hold elements of a single data type and support fast vectorised operations.
- Unlike numpy arrays, tensors take two extra arguments: `device` (whether the computation happens on CPU or GPU) and `requires_grad` (whether to track operations so that derivatives can be computed).
- Numpy underpins most of the classical data science stack (pandas, scikit-learn), whereas pytorch is mainly used for deep learning, which requires heavy matrix computation.

## 2.1 Optimisation example

$y=\sin(x)$ for $x = [-\pi,\pi]$. 

In [16]:
import numpy as np
import math

# Create random input and output data
x_np = np.linspace(-math.pi, math.pi, 2000)
y_np = np.sin(x_np)

### Optimisation by gradient descent in numpy

$y_{pred}=a + b x + c x^2 +d x^3$. We'll use torch's automatic differentiation to compute the gradient, and use gradient descent with a squared error loss function.

Technical note: we use the variable name `x_np` because its a numpy array, and we do the loss calculation manually in numpy here. We'll make a torch tensor version later called `x`, so that we can automate this with pytorch. 

In [17]:


# Randomly initialize weights
a = np.random.randn()
b = np.random.randn()
c = np.random.randn()
d = np.random.randn()

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    # y = a + b x + c x^2 + d x^3
    y_pred = a + b * x_np + c * x_np ** 2 + d * x_np ** 3

    # Compute and print loss
    loss = np.square(y_pred - y_np).sum()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y_np)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x_np).sum()
    grad_c = (grad_y_pred * x_np ** 2).sum()
    grad_d = (grad_y_pred * x_np ** 3).sum()

    # Update weights
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d

print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

99 28.827290177430683
199 22.368498488769212
299 18.00418020472676
399 15.052147255534264
499 13.05332011373848
599 11.698480760372316
699 10.77915934972357
799 10.1546749858459
899 9.72999961813785
999 9.440878653892376
1099 9.243820790298889
1199 9.109357659719379
1299 9.017501051291601
1399 8.954678339496628
1499 8.911663110597686
1599 8.882176412850232
1699 8.861940337148393
1799 8.84803698546566
1899 8.838473824076242
1999 8.83188865572017
Result: y = 0.0032887876299335255 + 0.854552150820291 x + -0.0005673705918259137 x^2 + -0.09301906662683923 x^3


### Optimisation by gradient descent in pytorch

Note that we have to use "tensors" for everything, as defined by the `torch.X` variable type.

In [18]:
import torch
import math


dtype = torch.float
device = torch.device("cpu")
# device = torch.device("cuda:0") # Uncomment this to run on GPU

# Create random input and output data
x = torch.linspace(-math.pi, math.pi, 2000, device=device, dtype=dtype)
y = torch.sin(x)


In [19]:

# Randomly initialize weights
a = torch.randn((), device=device, dtype=dtype)
b = torch.randn((), device=device, dtype=dtype)
c = torch.randn((), device=device, dtype=dtype)
d = torch.randn((), device=device, dtype=dtype)

learning_rate = 1e-6
for t in range(2000):
    # Forward pass: compute predicted y
    y_pred = a + b * x + c * x ** 2 + d * x ** 3

    # Compute and print loss
    loss = (y_pred - y).pow(2).sum().item()
    if t % 100 == 99:
        print(t, loss)

    # Backprop to compute gradients of a, b, c, d with respect to loss
    grad_y_pred = 2.0 * (y_pred - y)
    grad_a = grad_y_pred.sum()
    grad_b = (grad_y_pred * x).sum()
    grad_c = (grad_y_pred * x ** 2).sum()
    grad_d = (grad_y_pred * x ** 3).sum()

    # Update weights using gradient descent
    a -= learning_rate * grad_a
    b -= learning_rate * grad_b
    c -= learning_rate * grad_c
    d -= learning_rate * grad_d


print(f'Result: y = {a.item()} + {b.item()} x + {c.item()} x^2 + {d.item()} x^3')

99 599.1585693359375
199 401.35516357421875
299 269.92413330078125
399 182.56491088867188
499 124.4797592163086
599 85.84547424316406
699 60.13867950439453
799 43.027034759521484
899 31.63202476501465
999 24.040393829345703
1099 18.98044204711914
1199 15.606237411499023
1299 13.35495376586914
1399 11.852145195007324
1499 10.84836196899414
1599 10.177507400512695
1699 9.728875160217285
1799 9.428646087646484
1899 9.227605819702148
1999 9.092889785766602
Result: y = 0.008584746159613132 + 0.8708258271217346 x + -0.001481012674048543 x^2 + -0.09533385187387466 x^3


## 2.2 Automatic differentiation

Above we derived the gradients by hand. If a tensor is created with `requires_grad=True`, pytorch records every operation applied to it (a *computational graph*). Calling `.backward()` on a scalar result then fills in `.grad` for every such tensor.

We also vectorise the model: put the parameters in a single vector $w=(a,b,c,d)$ and the inputs in a *design matrix* $X$ with columns $1, x, x^2, x^3$, so the prediction is just `X @ w`.

Let's check autograd against our hand-derived gradient:

In [20]:
# Design matrix: columns 1, x, x^2, x^3
X = torch.stack([x**0, x, x**2, x**3], dim=1)

# One parameter vector w = (a, b, c, d), tracked by autograd
w = torch.randn(4, requires_grad=True)

loss = (X @ w - y).pow(2).sum()
loss.backward()   # computes d(loss)/dw and stores it in w.grad

# The gradient we derived by hand, in vectorised form
grad_manual = X.T @ (2.0 * (X @ w - y))

print("autograd:", w.grad)
print("manual:  ", grad_manual)

autograd: tensor([  10077.3574,  -22418.5977,   59030.3789, -146195.1094])
manual:   tensor([  10077.3574,  -22418.5977,   59030.3789, -146195.1094],
       grad_fn=<MvBackward0>)


## 2.3 Optimisers

Rather than updating the parameters ourselves, we hand them to an optimiser from `torch.optim`. Every iteration is then the same four steps:

1. `optimiser.zero_grad()`: clear the gradients from the previous step (otherwise they accumulate),
2. compute the loss,
3. `loss.backward()`: compute the gradients,
4. `optimiser.step()`: update the parameters.

`torch.optim.SGD` is exactly the gradient descent we wrote by hand above.

In [21]:
w = torch.randn(4, requires_grad=True)
optimiser = torch.optim.SGD([w], lr=1e-6)

for t in range(2000):
    optimiser.zero_grad()
    loss = (X @ w - y).pow(2).sum()
    loss.backward()
    optimiser.step()
    if t % 100 == 99:
        print(t, loss.item())

a, b, c, d = w.tolist()
print(f'Result: y = {a} + {b} x + {c} x^2 + {d} x^3')

99 2540.087158203125
199 1784.4814453125
299 1255.057861328125
399 883.9041748046875
499 623.56640625
599 440.8647766113281
699 312.5854187011719
799 222.47601318359375
899 159.15147399902344
999 114.63168334960938
1099 83.32027435302734
1199 61.290313720703125
1299 45.785179138183594
1399 34.86871337890625
1499 27.18056869506836
1599 21.764448165893555
1699 17.947839736938477
1799 15.257686614990234
1899 13.361059188842773
1999 12.023557662963867
Result: y = 0.058710549026727676 + 0.8456626534461975 x + -0.010128546506166458 x^2 + -0.09175460785627365 x^3


CHALLENGE: Replace `torch.optim.SGD` with `torch.optim.Adam`. What learning rate do you need? How quickly does it converge?

## 2.4 Beyond curve fitting: maximum likelihood

Nothing above was a neural network: we wrote down a loss and let pytorch differentiate it. The same recipe fits any model with a differentiable (log-)likelihood.

Here we fit a Normal distribution $N(\mu,\sigma^2)$ to data by maximum likelihood, i.e. by minimising the negative log-likelihood
$$
\mathrm{NLL}(\mu,\sigma) = \sum_i \left[\log\sigma + \frac{(x_i-\mu)^2}{2\sigma^2}\right] + \text{const}.
$$
We know the answer here (the sample mean and standard deviation), so we can check it worked.

$\sigma$ must be positive, so we optimise $\log\sigma$ instead, which can take any value. Reparameterising to remove constraints is a standard trick.

In [22]:
# Simulated data from N(3, 2^2)
data = torch.randn(1000) * 2 + 3

mu = torch.tensor(0.0, requires_grad=True)
log_sigma = torch.tensor(0.0, requires_grad=True)
optimiser = torch.optim.Adam([mu, log_sigma], lr=0.05)

for t in range(1000):
    optimiser.zero_grad()
    nll = (log_sigma + 0.5 * ((data - mu) / log_sigma.exp())**2).sum()
    nll.backward()
    optimiser.step()

print(f"Gradient descent: mu = {mu.item():.4f}, sigma = {log_sigma.exp().item():.4f}")
print(f"Closed form:      mu = {data.mean():.4f}, sigma = {data.std(correction=0):.4f}")

Gradient descent: mu = 2.9894, sigma = 2.0027
Closed form:      mu = 2.9894, sigma = 2.0027


CHALLENGE: Fit a mixture of two Normals, $p(x) = \pi N(\mu_1,\sigma_1^2) + (1-\pi) N(\mu_2,\sigma_2^2)$, in the same way. How do you keep $\pi$ between 0 and 1? Does it always find the same answer?

## Discussion

* What are the limits of automatic differentiation models? 
    - How would we fit physics models with them?
    - What about clustering models?